# CS 3500 - The Model 🧠

## Import Libraries 📚

In [17]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
import sys

device = torch.device("cpu")  # CPU only

sys.path.append('bolt-job')
from run import Model

## Read-In And Examine Cleaned Dataset 👀

In [18]:
combined_df = pd.read_csv('Cleaned_Dataset.csv')

# Split Data
def split_data(df, n_test=0.2, shuffle=True):
    if shuffle:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    test_size = int(n_test * len(df))
    train_size = len(df) - test_size

    train = df[:train_size]
    test = df[train_size:]
    return train, test

train, test = split_data(combined_df)
print("Length Of Train", len(train))
print("Length Of Test", len(test))

Length Of Train 189451
Length Of Test 47362


In [19]:
# Forgot to drop mo_codes --> Drop mo_codes as data has been extracted
train.drop(columns = "mo_codes", axis = 1, inplace = True)
train.head()

,area_code,reporting_district,crime_part,crime_code,victim_age,premise_code,weapon_code,days_to_holiday,report_year,report_month,...,occ_time_interval_06:01-12:00,occ_time_interval_12:01-18:00,occ_time_interval_18:01-24:00,occurrence_time_of_day_Afternoon,occurrence_time_of_day_Early Morning,occurrence_time_of_day_Evening,occurrence_time_of_day_Late Night,occurrence_time_of_day_Morning,occurrence_time_of_day_Night,Status
0,19,1959,1,331,48,101.0,0.0,8,2023,7,...,False,False,True,False,False,False,False,False,True,2
1,20,2026,1,230,26,101.0,302.0,10,2023,12,...,False,False,False,False,False,False,True,False,False,2
2,4,412,1,330,27,504.0,0.0,19,2024,6,...,False,False,True,False,False,False,False,False,True,2
3,19,1974,2,930,35,502.0,511.0,14,2023,6,...,False,False,True,False,False,False,False,False,True,0
4,3,375,2,354,50,501.0,0.0,17,2023,2,...,False,False,False,False,False,False,True,False,False,2


In [20]:
test.drop(columns = "mo_codes", axis = 1, inplace = True)
test.head()

,area_code,reporting_district,crime_part,crime_code,victim_age,premise_code,weapon_code,days_to_holiday,report_year,report_month,...,occ_time_interval_06:01-12:00,occ_time_interval_12:01-18:00,occ_time_interval_18:01-24:00,occurrence_time_of_day_Afternoon,occurrence_time_of_day_Early Morning,occurrence_time_of_day_Evening,occurrence_time_of_day_Late Night,occurrence_time_of_day_Morning,occurrence_time_of_day_Night,Status
189451,1,155,2,626,46,502.0,400.0,7,2023,2,...,False,True,False,False,False,True,False,False,False,1
189452,9,985,1,330,40,101.0,0.0,2,2023,10,...,False,True,False,False,False,True,False,False,False,2
189453,9,923,2,664,80,102.0,0.0,15,2023,2,...,False,False,True,False,False,True,False,False,False,1
189454,13,1331,1,230,53,101.0,106.0,6,2023,6,...,False,False,False,False,False,False,True,False,False,2
189455,13,1317,2,624,19,935.0,400.0,1,2023,10,...,False,True,False,True,False,False,False,False,False,2


In [21]:
train.info() # All datatypes float, int, boolean --> We are now able to move on

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189451 entries, 0 to 189450
Columns: 600 entries, area_code to Status
dtypes: bool(560), float64(2), int64(38)
memory usage: 159.0 MB


## Define Features/Target And Perform Light Preprocessing ♻️
* We Are Scaling Features since the dataset contains 0 values to values in the millions! This can affect the performance of our model.
   

In [22]:
# Define target and features - Train
train_features = train.drop(columns="Status")
train_target = train["Status"]

# Define target and features - Test
test_features = test.drop(columns="Status")
test_target = test["Status"]

# Scale Features
feature_scaler = PowerTransformer(method='yeo-johnson')
scaled_train_features = feature_scaler.fit_transform(train_features)
scaled_test_features = feature_scaler.fit_transform(test_features)

## Create DataLoaders (*Convert Pandas DF To Tensors*) ⚙️

In [23]:
class Crime_Dataset(Dataset):
    def __init__(self, features, targets):
        # Conver to Torch Tensors
        self.x = torch.tensor(features, dtype = torch.float32)
        self.y = torch.tensor(targets.values, dtype = torch.float32)

    # Define mandatory length method
    def __len__(self):
        return len(self.y)
    
    # Define mandatory get item method
    def __getitem__(self, index):
        return self.x[index], self.y[index]

# Instantiate Class With Data
train_df = Crime_Dataset(scaled_train_features, train_target)
test_df = Crime_Dataset(scaled_test_features, test_target)

test_loader = DataLoader(test_df, batch_size=32, shuffle=False)

In [24]:
model = Model()
model.load_state_dict(torch.load('../bolt-job/model_final.pt', map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = torch.sigmoid(model(X_batch))
        preds = (outputs > 0.5).long().squeeze(1)
        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds, digits=4))

for i in range(5):
    print(f"Prediction: {all_preds[i]} \t Actual: {all_labels[i]}")

FileNotFoundError: [Errno 2] No such file or directory: '../bolt-job/model_final.pt'

In [ ]:
import matplotlib.pyplot as plt

unique_labels = ['No Arrest', 'Arrest']
pred_counts = [all_preds.count(0), all_preds.count(1)]
actual_counts = [all_labels.count(0), all_labels.count(1)]

x = np.arange(len(unique_labels))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, actual_counts, width, label='Actual')
ax.bar(x + width/2, pred_counts, width, label='Predicted')

ax.set_ylabel('Count')
ax.set_title('Predicted vs Actual Class Distribution')
ax.set_xticks(x)
ax.set_xticklabels(unique_labels)
ax.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=['No Arrest', 'Arrest'], yticklabels=['No Arrest', 'Arrest'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Collect probabilities and true labels
all_probs = []
all_labels = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = torch.sigmoid(model(X_batch))  # probabilities
        all_probs.extend(outputs.squeeze(1).numpy())
        all_labels.extend(y_batch.numpy())


fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D plotting
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce to 3D
pca_3d = PCA(n_components=4)
X_pca_3d = pca_3d.fit_transform(scaled_test_features)

# Get actuals and predicted
actuals = np.array(test_target)
with torch.no_grad():
    probs = torch.sigmoid(model(torch.tensor(scaled_test_features, dtype=torch.float32)))
    preds = (probs > 0.5).long().squeeze(1).numpy()
correct = preds == actuals

# Plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Correct predictions
ax.scatter(X_pca_3d[correct, 0], X_pca_3d[correct, 1], X_pca_3d[correct, 2],
           c=actuals[correct], cmap='coolwarm', s=10, label='Correct', alpha=0.6)

# Incorrect predictions
ax.scatter(X_pca_3d[~correct, 0], X_pca_3d[~correct, 1], X_pca_3d[~correct, 2],
           c=actuals[~correct], cmap='coolwarm', edgecolors='k', s=25, label='Incorrect')

ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
ax.set_zlabel('PCA 3')
ax.set_title('3D PCA Projection with Prediction Accuracy')
ax.legend()
plt.show()